# Brain Tumor MRI Classification with OpenCV + a CNN

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vignxhhh/brain-tumor-mri-cnn/blob/main/brain_tumor_cnn.ipynb) [![GitHub](https://img.shields.io/badge/GitHub-brain--tumor--mri--cnn-181717?logo=github)](https://github.com/vignxhhh/brain-tumor-mri-cnn)

Classifies brain MRI slices into **4 classes**: glioma tumor, meningioma tumor, pituitary tumor, and normal (no tumor).

**Workflow**
1. Load the data and group augmented copies with their original scan
2. Split into train / validation / test by scan ID (no leakage)
3. **OpenCV preprocessing**: grayscale → denoise → find the head outline → crop → CLAHE contrast → resize
4. Explore the data (class counts, image sizes, sample images)
5. Build the CNN
6. Train it
7. Evaluate it on the held-out test set
8. Predict new images

**Dataset layout expected** ([Brain MRI images dataset](https://www.kaggle.com/datasets/babaraliuser/brain-mri-images-dataset) on Kaggle):
```
archive/mri/
    glioma_tumor/      G_1.jpg, G_1_BR_.jpg, G_1_HF_.jpg, ...
    meningioma_tumor/  M_1.jpg, ...
    normal/            N_1.jpg, ...
    pituitary_tumor/   P_1.jpg, P_1_BR_.jpg, ...
```
Each original scan (e.g. `P_100.jpg`) has 6 augmented copies with suffixes
`_BR_` (brightness), `_DA_`, `_HF_` (horizontal flip), `_RO_` (rotation), `_SP_` (salt-and-pepper noise),
`_VF_` (vertical flip).

**Important — avoiding data leakage:** if you split files randomly, `P_100_HF_.jpg` can land in training while
`P_100.jpg` lands in the test set. The model then "recognises" a flipped copy of an image it already saw, and the
test accuracy is inflated. This notebook splits by **original scan ID**, so every version of a scan stays in
the same split.

In [ ]:
# Setup: locate the dataset (works on Kaggle, Google Colab, or a local machine)
import os, glob
hits = glob.glob("/kaggle/input/**/glioma_tumor", recursive=True) or glob.glob("**/glioma_tumor", recursive=True)
if hits:
    os.environ.setdefault("DATA_DIR", os.path.dirname(hits[0]))
    print("Dataset found at", os.environ["DATA_DIR"])
else:
    print("Dataset not found. Download it from https://www.kaggle.com/datasets/babaraliuser/brain-mri-images-dataset,\n"
          "unzip it next to this notebook, or set the DATA_DIR environment variable to the folder holding the 4 class folders.")
os.environ.setdefault("OUT_DIR", "/kaggle/working/outputs" if os.path.isdir("/kaggle/working") else "outputs")

In [28]:
import os, re, random
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

import cv2                                   # OpenCV — image preprocessing
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ---- Configuration ---------------------------------------------------------
DATA_DIR   = Path(os.environ.get("DATA_DIR", "archive/mri"))   # folder holding the 4 class folders
IMG_SIZE   = int(os.environ.get("IMG_SIZE", 224))              # images are resized to IMG_SIZE x IMG_SIZE
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", 32))
EPOCHS     = int(os.environ.get("EPOCHS", 40))                 # early stopping usually ends sooner
LR         = 1e-3
OUT_DIR    = Path(os.environ.get("OUT_DIR", "outputs")); OUT_DIR.mkdir(exist_ok=True)
CACHE_DIR  = OUT_DIR / f"preprocessed_{IMG_SIZE}"              # OpenCV output is saved here once and reused

CLASS_NAMES = ["glioma_tumor", "meningioma_tumor", "normal", "pituitary_tumor"]
NUM_CLASSES = len(CLASS_NAMES)
print("TensorFlow", tf.__version__, "| OpenCV", cv2.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow 2.21.0 | OpenCV 4.14.0 | GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


## 1. Load file paths and group augmented copies with their original

In [29]:
AUG_SUFFIX = re.compile(r"_(BR|DA|HF|RO|SP|VF)_$", re.IGNORECASE)

def scan_id(path: Path) -> str:
    """'P_100_HF_.jpg' -> 'pituitary_tumor/P_100' (same ID for the original and all its augmentations)."""
    return f"{path.parent.name}/{AUG_SUFFIX.sub('', path.stem)}"

def is_original(path: Path) -> bool:
    return AUG_SUFFIX.search(path.stem) is None

paths, labels = [], []
for idx, cls in enumerate(CLASS_NAMES):
    files = sorted(p for p in (DATA_DIR / cls).glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
    assert files, f"No images found in {DATA_DIR / cls}"
    paths += files; labels += [idx] * len(files)

paths, labels = np.array(paths), np.array(labels)
groups = np.array([scan_id(p) for p in paths])
print(f"{len(paths)} images from {len(set(groups))} original scans")
for i, c in enumerate(CLASS_NAMES):
    print(f"  {c:18s} {np.sum(labels == i):5d} images, {len(set(groups[labels == i])):4d} scans")

21672 images from 3096 original scans
  glioma_tumor        6307 images,  901 scans
  meningioma_tumor    6391 images,  913 scans
  normal              3066 images,  438 scans
  pituitary_tumor     5908 images,  844 scans


## 2. Split by scan ID — 70% train / 15% validation / 15% test (stratified by class)

In [30]:
uniq_groups = np.array(sorted(set(groups)))
group_label = np.array([CLASS_NAMES.index(g.split("/")[0]) for g in uniq_groups])

g_train, g_tmp, _, y_tmp = train_test_split(uniq_groups, group_label, test_size=0.30,
                                            stratify=group_label, random_state=SEED)
g_val, g_test = train_test_split(g_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)

def subset(gs):
    m = np.isin(groups, gs)
    return paths[m], labels[m]

X_train, y_train = subset(g_train)
X_val,   y_val   = subset(g_val)
X_test,  y_test  = subset(g_test)

# Sanity check: no scan appears in more than one split
assert not (set(g_train) & set(g_val) or set(g_train) & set(g_test) or set(g_val) & set(g_test))
print(f"train {len(X_train)} | val {len(X_val)} | test {len(X_test)} images  (no scan shared across splits)")

train 15169 | val 3248 | test 3255 images  (no scan shared across splits)


## 3. Image preprocessing with OpenCV
Raw scans contain a lot of black background, come in different sizes (e.g. 512×512 and 224×224), some have
salt-and-pepper noise (`_SP_`), and contrast varies (`_BR_` brighter, `_DA_` darker). OpenCV cleans this up
**before** the CNN sees the images:

| Step | OpenCV function | Why |
|---|---|---|
| 1. Read as grayscale | `cv2.imread(path, cv2.IMREAD_GRAYSCALE)` | MRI is single-channel; 3 identical RGB channels just waste memory |
| 2. Remove noise | `cv2.medianBlur` | Median filtering removes salt-and-pepper dots while keeping edges sharp |
| 3. Blur for masking | `cv2.GaussianBlur` | Smooths the image so the head outline comes out as one clean shape |
| 4. Threshold | `cv2.threshold` | Pixels brighter than the background level + 15 become white (head), the rest black |
| 5. Clean the mask | `cv2.morphologyEx` (open, close) | *Open* removes small specks; *close* fills gaps so the head is one solid blob |
| 6. Find the head outline | `cv2.findContours` | The largest external contour is the head |
| 7. Crop | `cv2.boundingRect` + array slicing | Cuts away the empty background so the head fills the frame |
| 8. Boost contrast | `cv2.createCLAHE` | Local histogram equalisation makes tumor boundaries stand out |
| 9. Resize | `cv2.resize(..., interpolation=cv2.INTER_AREA)` | Every image becomes IMG_SIZE×IMG_SIZE; `INTER_AREA` is best for shrinking |

The background level is measured from the image border instead of using a fixed number, so the brightened
(`_BR_`) images, whose background is grey rather than black, are still cropped correctly.
If no head outline is found, the full image is used rather than failing.

In [31]:


def find_head_box(gray):
    """Return (x, y, w, h) of the head, plus the mask and contour used to find it (None if not found)."""
    blur = cv2.GaussianBlur(cv2.medianBlur(gray, 5), (5, 5), 0)
    border = np.concatenate([blur[:5].ravel(), blur[-5:].ravel(), blur[:, :5].ravel(), blur[:, -5:].ravel()])
    thresh = float(np.median(border)) + 15                      # background brightness + a margin
    _, mask = cv2.threshold(blur, thresh, 255, cv2.THRESH_BINARY)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)   # remove specks
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)  # fill gaps
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None, mask, None
    head = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(head)
    if w * h < 0.05 * gray.size:                                 # too small to be a head -> don't trust it
        return None, mask, head
    return (x, y, w, h), mask, head

def preprocess_cv(path, size=IMG_SIZE):
    """Full OpenCV pipeline: path -> cleaned, cropped, contrast-enhanced, resized uint8 grayscale image."""
    gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise ValueError(f"OpenCV could not read {path}")
    clean = cv2.medianBlur(gray, 3)                              # light denoise for the final image
    box, _, _ = find_head_box(gray)
    if box is not None:
        x, y, w, h = box
        pad = int(0.02 * max(gray.shape))                        # small margin so the skull edge isn't clipped
        clean = clean[max(0, y - pad):y + h + pad, max(0, x - pad):x + w + pad]
    clean = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(clean)
    return cv2.resize(clean, (size, size), interpolation=cv2.INTER_AREA)

### 3a. See each OpenCV step on one image per class

In [ ]:
def show_steps(path, axes):
    gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    box, mask, head = find_head_box(gray)
    outline = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    if head is not None:
        cv2.drawContours(outline, [head], -1, (0, 255, 0), 2)
    if box is not None:
        x, y, w, h = box; cv2.rectangle(outline, (x, y), (x + w, y + h), (255, 0, 0), 2)
    final = preprocess_cv(path)
    for ax, img, title in zip(axes, [gray, mask, outline, final],
                              ["1. original (grayscale)", "2. threshold mask", "3. contour + crop box",
                               "4. cropped + CLAHE + resized"]):
        ax.imshow(img, cmap="gray"); ax.set_title(title, fontsize=9); ax.axis("off")

fig, axes = plt.subplots(NUM_CLASSES, 4, figsize=(13, 3.2 * NUM_CLASSES))
for i in range(NUM_CLASSES):
    example = next(p for p in X_train[y_train == i] if is_original(p)) if any(
        is_original(p) for p in X_train[y_train == i]) else X_train[y_train == i][0]
    show_steps(example, axes[i])
    axes[i, 0].set_ylabel(CLASS_NAMES[i]); axes[i, 0].set_title(f"{CLASS_NAMES[i]}\n1. original (grayscale)", fontsize=9)
plt.tight_layout(); plt.savefig(OUT_DIR / "opencv_steps.png", dpi=110); plt.show()

### 3b. Run the OpenCV pipeline on every image (once)
Preprocessing ~20,000 images takes a few minutes, so the results are saved to `CACHE_DIR` as PNG files and
reused on later runs. Training then reads the already-clean images, which keeps each epoch fast.

In [ ]:
def cached_path(p: Path) -> Path:
    return CACHE_DIR / p.parent.name / (p.stem + ".png")

def preprocess_and_save(p: Path):
    out = cached_path(p)
    if not out.exists():
        out.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(out), preprocess_cv(p))
    return out

with ThreadPoolExecutor(max_workers=os.cpu_count()) as pool:
    done = list(pool.map(preprocess_and_save, paths))
print(f"{len(done)} preprocessed images in {CACHE_DIR}")

P_train = np.array([cached_path(p) for p in X_train])
P_val   = np.array([cached_path(p) for p in X_val])
P_test  = np.array([cached_path(p) for p in X_test])

## 4. Explore the data

In [ ]:
# Class balance (images and unique scans per class)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
img_counts  = [np.sum(labels == i) for i in range(NUM_CLASSES)]
scan_counts = [len(set(groups[labels == i])) for i in range(NUM_CLASSES)]
ax[0].bar(CLASS_NAMES, img_counts, color="#4C72B0"); ax[0].set_title("Images per class")
ax[1].bar(CLASS_NAMES, scan_counts, color="#55A868"); ax[1].set_title("Original scans per class")
for a in ax: a.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.savefig(OUT_DIR / "class_balance.png", dpi=110); plt.show()

# Raw image sizes, read with OpenCV (sample of up to 500 to keep it quick)
sample = np.random.choice(paths, size=min(500, len(paths)), replace=False)
sizes = Counter(cv2.imread(str(p), cv2.IMREAD_GRAYSCALE).shape for p in sample)
print("Raw image sizes (height, width) in a sample:", dict(sizes.most_common(8)))
print(f"After OpenCV preprocessing every image is {IMG_SIZE}x{IMG_SIZE} grayscale.")

# Raw vs. preprocessed, two original scans per class
fig, axes = plt.subplots(2, 2 * NUM_CLASSES, figsize=(18, 5))
for i in range(NUM_CLASSES):
    picks = [p for p in X_train[y_train == i] if is_original(p)][:2] or list(X_train[y_train == i][:2])
    for row, p in enumerate(picks[:2]):
        axes[row, 2 * i].imshow(cv2.imread(str(p), cv2.IMREAD_GRAYSCALE), cmap="gray")
        axes[row, 2 * i + 1].imshow(cv2.imread(str(cached_path(p)), cv2.IMREAD_GRAYSCALE), cmap="gray")
        if row == 0:
            axes[row, 2 * i].set_title(f"{CLASS_NAMES[i]}\nraw", fontsize=9)
            axes[row, 2 * i + 1].set_title("OpenCV", fontsize=9)
for a in axes.flat: a.axis("off")
plt.tight_layout(); plt.savefig(OUT_DIR / "samples_raw_vs_opencv.png", dpi=110); plt.show()

## 5. tf.data input pipeline
Reads the OpenCV-preprocessed PNGs (1 channel) and scales pixels to [0, 1]. The dataset is already augmented on
disk, so only a light random flip/rotation/zoom is added on the fly inside the model during training.

In [ ]:
def load_image(path, label):
    img = tf.io.decode_png(tf.io.read_file(path), channels=1)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))           # no-op unless IMG_SIZE changed
    return tf.cast(img, tf.float32) / 255.0, label

def make_ds(P, y, training):
    ds = tf.data.Dataset.from_tensor_slices((P.astype(str), y))
    if training:
        ds = ds.shuffle(len(P), seed=SEED, reshuffle_each_iteration=True)
    return ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(P_train, y_train, training=True)
val_ds   = make_ds(P_val, y_val, training=False)
test_ds  = make_ds(P_test, y_test, training=False)

# Class weights compensate for any imbalance (e.g. fewer no_tumor scans)
cw = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=y_train)
class_weight = dict(enumerate(cw))
print("class weights:", {CLASS_NAMES[k]: round(v, 2) for k, v in class_weight.items()})

## 6. CNN architecture
Four convolution blocks (Conv → BatchNorm → ReLU, twice, then MaxPool + Dropout). Filters double each block
(32 → 64 → 128 → 256) so the network learns edges first, then textures, then tumor-shaped structures.
Global average pooling replaces a huge Flatten layer, which keeps the parameter count small and reduces overfitting.
The input has **1 channel** because OpenCV produced grayscale images.

In [ ]:
def conv_block(x, filters, dropout):
    for _ in range(2):
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)
    return layers.Dropout(dropout)(x)

def build_model():
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 1))
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomRotation(0.05)(x)
    x = layers.RandomZoom(0.1)(x)
    x = conv_block(x, 32, 0.10)
    x = conv_block(x, 64, 0.15)
    x = conv_block(x, 128, 0.20)
    x = conv_block(x, 256, 0.25)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="brain_tumor_cnn")

model = build_model()
model.compile(optimizer=keras.optimizers.Adam(LR),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 7. Train
- **EarlyStopping** stops when validation loss stops improving and restores the best weights.
- **ReduceLROnPlateau** halves the learning rate when progress stalls.
- **ModelCheckpoint** saves the best model to disk.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    keras.callbacks.ModelCheckpoint(str(OUT_DIR / "best_model.keras"), monitor="val_loss", save_best_only=True),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                    class_weight=class_weight, callbacks=callbacks, verbose=2)

## 8. Training curves

In [ ]:
h = history.history
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(h["accuracy"], label="train"); ax[0].plot(h["val_accuracy"], label="validation")
ax[0].set_title("Accuracy"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(h["loss"], label="train"); ax[1].plot(h["val_loss"], label="validation")
ax[1].set_title("Loss"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.savefig(OUT_DIR / "training_curves.png", dpi=110); plt.show()

## 9. Evaluate on the held-out test set
Reported twice: on every test image, and on the **original scans only** (no augmented copies), which is the
most honest estimate of performance on new patients' scans.

In [ ]:
probs  = model.predict(test_ds, verbose=0)
y_pred = probs.argmax(axis=1)

def report(mask, title):
    acc = (y_pred[mask] == y_test[mask]).mean()
    print(f"\n=== {title}: accuracy {acc:.4f} on {mask.sum()} images ===")
    print(classification_report(y_test[mask], y_pred[mask], labels=range(NUM_CLASSES),
                                target_names=CLASS_NAMES, digits=4, zero_division=0))
    return acc

all_mask  = np.ones(len(y_test), bool)
orig_mask = np.array([is_original(p) for p in X_test])
report(all_mask, "Test set (all images)")
if orig_mask.any():
    report(orig_mask, "Test set (original scans only)")

cm = confusion_matrix(y_test, y_pred, labels=range(NUM_CLASSES))
plt.figure(figsize=(6.5, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion matrix — test set")
plt.xticks(rotation=30, ha="right"); plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_matrix.png", dpi=110); plt.show()

## 10. Example predictions (green = correct, red = wrong)

In [ ]:
idx = np.random.choice(len(X_test), size=min(12, len(X_test)), replace=False)
fig, axes = plt.subplots(3, 4, figsize=(13, 10))
for ax, i in zip(axes.flat, idx):
    ax.imshow(cv2.imread(str(P_test[i]), cv2.IMREAD_GRAYSCALE), cmap="gray"); ax.axis("off")
    ok = y_pred[i] == y_test[i]
    ax.set_title(f"true: {CLASS_NAMES[y_test[i]]}\npred: {CLASS_NAMES[y_pred[i]]} ({probs[i].max():.0%})",
                 color="green" if ok else "red", fontsize=9)
plt.tight_layout(); plt.savefig(OUT_DIR / "sample_predictions.png", dpi=110); plt.show()

## 11. Predict a single new image
A new scan goes through exactly the same OpenCV preprocessing as the training images before the model sees it.

In [ ]:
def predict_image(path, model=model):
    img = preprocess_cv(path).astype(np.float32) / 255.0       # OpenCV pipeline, then scale to [0, 1]
    p = model.predict(img[None, ..., None], verbose=0)[0]      # add batch and channel dimensions
    return CLASS_NAMES[int(p.argmax())], {c: float(v) for c, v in zip(CLASS_NAMES, p)}

label, scores = predict_image(X_test[0])
print("File:", X_test[0]); print("Predicted:", label); print({k: round(v, 3) for k, v in scores.items()})

model.save(OUT_DIR / "brain_tumor_cnn_final.keras")
print("Saved model to", OUT_DIR / "brain_tumor_cnn_final.keras")